# Agentic AI I: ReAct & Tool Orchestration

Day 3 of Week 1. A single LLM call answers a question. An agent solves a task — it decides what information to gather, which tools to invoke, and when it has enough to respond. This notebook builds the ReAct (Reasoning + Acting) loop from scratch: a while-loop that alternates between the model reasoning about its next step and executing a tool, continuing until the model decides it has an answer. We implement a small tool registry, wire up four financial tools, and build a single-agent system that can answer multi-step questions about SEC filings, portfolio positions, and market data without any framework.

The ReAct pattern was introduced by Yao et al. (2022) as a way to interleave chain-of-thought reasoning with action execution. The key insight is that reasoning about what to do and doing it should alternate — the model's reasoning in one step informs the action it takes, and the result of that action informs the next round of reasoning.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## From Single Call to Agent Loop

**Single call vs. agent.** A single LLM call is a function: input text → output text. The model can only use the knowledge encoded in its weights plus whatever we put in the context window. An agent is an iterative process: the model can request new information by calling tools, and the results of those tool calls become part of the context for the next reasoning step.

<br>

**Why iteration matters.** Consider the question: "Is this portfolio over-exposed to interest rate risk?" To answer this, the agent must (1) fetch the portfolio positions, (2) fetch current interest rates, (3) compute the duration-weighted exposure, and (4) compare against the risk limit. Each step's output determines what the next step should do. A single LLM call cannot do this — it has no way to fetch live data or compute with precise arithmetic.

<br>

**The ReAct loop.** The loop alternates between two actions:

- **Reason**: the model receives the current context (original question + all previous tool results) and either (a) calls a tool by emitting a `tool_calls` response, or (b) emits a final text answer (`stop` finish reason).
- **Act**: when the model emits a `tool_calls` response, we execute the requested tool(s), append the results to the message history, and call the model again.

The loop terminates when the model decides it has enough information to answer, or when `max_steps` is exceeded. The message history grows with each step — this is the agent's in-context memory.

:::{.callout-note}
OpenAI's tool-calling API handles the ReAct pattern natively: the model emits a `tool_calls` list in its response when it wants to call tools, and we append `tool` role messages with the results. The model sees this entire exchange in its context on the next call.

:::

## Tool Registry

We need a uniform interface for tools: each tool has a name, a description (which the model reads to decide whether to use it), a JSON Schema for its parameters, and a Python callable. We wrap these in a `Tool` dataclass and a `ToolRegistry` that translates the registry into the list-of-dicts format the OpenAI API expects.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Any


@dataclass
class Tool:
    name: str
    description: str
    parameters: dict  # JSON Schema object
    fn: Callable[..., Any]


class ToolRegistry:
    """Registry translating Tool objects into OpenAI function-calling format."""

    def __init__(self):
        self._tools: dict[str, Tool] = {}

    def register(self, tool: Tool) -> None:
        self._tools[tool.name] = tool

    def get_schema(self) -> list[dict]:  # <1>
        """Return the list of tool dicts for the OpenAI tools parameter."""
        return [
            {
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.parameters,
                },
            }
            for t in self._tools.values()
        ]

    def call(self, name: str, arguments: str) -> str:  # <2>
        """Execute the named tool with JSON-encoded arguments."""
        if name not in self._tools:
            return f"Error: unknown tool '{name}'"
        kwargs = json.loads(arguments)
        try:
            result = self._tools[name].fn(**kwargs)
            return json.dumps(result) if not isinstance(result, str) else result
        except Exception as e:
            return f"Error executing {name}: {e}"

1. `get_schema` produces the exact format the OpenAI `tools` parameter expects — a list of objects each with `type: "function"` and a nested `function` dict containing `name`, `description`, and `parameters` (JSON Schema).
2. `call` deserializes the JSON-encoded `arguments` string that the model emits, invokes the Python callable, and serializes the result back to a string for the `tool` role message.

We register four financial tools. The corpus from [notebook 03](/courses/llm-eng/03-rag-concepts.html) serves as the filing database:

In [ ]:
# The 10-sentence financial corpus, keyed by section
FILING_SECTIONS = {
    "interest_rate_risk": "Interest rate risk represents one of the most significant market risks facing the firm. A 100 basis point increase in interest rates would reduce the fair value of our fixed-rate debt portfolio by approximately $2.3 billion.",
    "credit_risk": "Credit risk arises from the potential that a counterparty will fail to perform its obligations. We manage credit risk through diversification, collateral requirements, and credit limits by counterparty.",
    "operational_risk": "Operational risk includes the risk of loss resulting from inadequate or failed internal processes, people, systems, or external events, including cybersecurity threats and technology failures.",
    "net_revenues": "Net revenues for the fiscal year were $47.4 billion, an increase of 8% compared to the prior year. The increase was driven primarily by higher net interest income reflecting the rising interest rate environment.",
    "investment_banking": "Investment banking revenues decreased 23% to $6.1 billion, reflecting lower advisory fees amid reduced M&A activity and a challenging environment for equity and debt underwriting.",
    "return_on_equity": "Return on equity for the year was 12.4%, compared to 15.1% in the prior year. Book value per share increased to $312.50, up from $290.20.",
    "cet1": "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and our internal target of 13%.",
    "lcr": "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the regulatory requirement of 100%. Our high-quality liquid assets totaled $280 billion at year-end.",
    "guidance": "Looking ahead to fiscal 2025, management expects continued revenue growth in the range of 4-6%, supported by a stable rate environment and improving capital markets activity.",
    "shareholder_returns": "We plan to return $8 billion to shareholders through dividends and share repurchases in fiscal 2025, subject to regulatory approval and market conditions.",
}


def get_filing_excerpt(ticker: str, section: str) -> str:
    """Retrieve a passage from the 10-K filing for the given section."""
    section = section.lower().replace(" ", "_")
    if section in FILING_SECTIONS:
        return f"[{ticker} 10-K] {FILING_SECTIONS[section]}"
    available = list(FILING_SECTIONS.keys())
    return f"Section '{section}' not found. Available: {available}"


def calculate_var(
    portfolio_value: float,
    volatility: float,
    confidence: float = 0.95,
) -> dict:
    """Compute parametric Value at Risk. VaR = portfolio_value * volatility * z_alpha."""
    from scipy.stats import norm
    z = norm.ppf(confidence)  # <1>
    var = portfolio_value * volatility * z
    return {
        "portfolio_value": portfolio_value,
        "volatility": volatility,
        "confidence": confidence,
        "z_alpha": round(z, 4),
        "var": round(var, 2),
        "var_pct": round(var / portfolio_value * 100, 2),
    }


def get_bond_price(
    face_value: float,
    coupon_rate: float,
    ytm: float,
    periods: int,
) -> dict:
    """Price a bond using the standard discounted cash flow formula."""
    coupon = face_value * coupon_rate
    # Price = sum of discounted coupons + discounted face value
    pv_coupons = sum(coupon / (1 + ytm) ** t for t in range(1, periods + 1))
    pv_face = face_value / (1 + ytm) ** periods
    price = pv_coupons + pv_face
    return {
        "face_value": face_value,
        "coupon_rate": coupon_rate,
        "ytm": ytm,
        "periods": periods,
        "price": round(price, 2),
        "premium_discount": round(price - face_value, 2),
    }


REGULATORY_THRESHOLDS = {
    "cet1": {"minimum": 0.045, "description": "Common Equity Tier 1 capital ratio minimum"},
    "lcr": {"minimum": 1.00, "description": "Liquidity Coverage Ratio minimum"},
    "tier1": {"minimum": 0.060, "description": "Tier 1 capital ratio minimum"},
    "total_capital": {"minimum": 0.080, "description": "Total capital ratio minimum"},
    "leverage": {"minimum": 0.030, "description": "Leverage ratio minimum"},
}


def lookup_regulatory_threshold(metric_name: str) -> dict:
    """Return the regulatory minimum for a given capital or liquidity metric."""
    key = metric_name.lower().replace(" ", "_").replace("-", "_")
    if key in REGULATORY_THRESHOLDS:
        return REGULATORY_THRESHOLDS[key]
    return {"error": f"Unknown metric '{metric_name}'. Known: {list(REGULATORY_THRESHOLDS.keys())}"}

1. The $z_{\alpha}$ quantile of the standard normal is the number of standard deviations corresponding to the confidence level $\alpha$. For $\alpha = 0.95$, $z_{0.95} \approx 1.645$. The parametric VaR formula assumes portfolio returns are normally distributed — a simplification that underestimates tail risk for heavy-tailed financial returns but is standard for back-of-envelope calculations.

We register all four tools with their JSON Schema parameter definitions:

In [ ]:
registry = ToolRegistry()

registry.register(Tool(
    name="get_filing_excerpt",
    description="Retrieve a passage from a company's 10-K SEC filing by section name.",
    parameters={
        "type": "object",
        "properties": {
            "ticker": {"type": "string", "description": "Stock ticker symbol, e.g. GS"},
            "section": {
                "type": "string",
                "description": "Section name: cet1, lcr, investment_banking, net_revenues, guidance, return_on_equity, interest_rate_risk, credit_risk, operational_risk, shareholder_returns",
            },
        },
        "required": ["ticker", "section"],
    },
    fn=get_filing_excerpt,
))

registry.register(Tool(
    name="calculate_var",
    description="Calculate parametric Value at Risk (VaR) for a portfolio.",
    parameters={
        "type": "object",
        "properties": {
            "portfolio_value": {"type": "number", "description": "Portfolio value in USD"},
            "volatility": {"type": "number", "description": "Annual volatility as a decimal (e.g. 0.15 for 15%)"},
            "confidence": {"type": "number", "description": "Confidence level, default 0.95"},
        },
        "required": ["portfolio_value", "volatility"],
    },
    fn=calculate_var,
))

registry.register(Tool(
    name="get_bond_price",
    description="Price a fixed-coupon bond given face value, coupon rate, yield to maturity, and number of periods.",
    parameters={
        "type": "object",
        "properties": {
            "face_value": {"type": "number"},
            "coupon_rate": {"type": "number", "description": "Annual coupon as decimal (e.g. 0.05 for 5%)"},
            "ytm": {"type": "number", "description": "Yield to maturity as decimal"},
            "periods": {"type": "integer", "description": "Number of coupon periods"},
        },
        "required": ["face_value", "coupon_rate", "ytm", "periods"],
    },
    fn=get_bond_price,
))

registry.register(Tool(
    name="lookup_regulatory_threshold",
    description="Look up the regulatory minimum for a capital or liquidity metric (CET1, LCR, Tier1, etc.).",
    parameters={
        "type": "object",
        "properties": {
            "metric_name": {"type": "string", "description": "Metric name, e.g. cet1, lcr, leverage"},
        },
        "required": ["metric_name"],
    },
    fn=lookup_regulatory_threshold,
))

print(f"Registered {len(registry._tools)} tools: {list(registry._tools.keys())}")

## The ReAct Loop

The loop is a while-loop over the OpenAI chat completion API. On each iteration, we call the API with the current message history and the tool schemas. If the model wants to call a tool (`finish_reason == "tool_calls"`), we execute each requested tool and append the results. If the model is ready to answer (`finish_reason == "stop"`), we return the final content. If we exceed `max_steps`, we raise `AgentTimeoutError`.

In [ ]:
class AgentTimeoutError(Exception):
    pass


def run_agent(
    question: str,
    registry: ToolRegistry,
    llm: LLMClient,
    max_steps: int = 10,
    verbose: bool = False,
) -> str:
    """Run the ReAct agent loop until the model emits a final answer."""
    client = openai.OpenAI()
    messages = [
        {
            "role": "system",
            "content": (
                "You are a financial analyst assistant. "
                "Use the available tools to gather information, then provide a concise, "
                "cited answer. Always verify regulatory thresholds before stating compliance."
            ),
        },
        {"role": "user", "content": question},
    ]

    for step in range(max_steps):
        resp = client.chat.completions.create(  # <1>
            model=llm.model,
            messages=messages,
            tools=registry.get_schema(),
            temperature=llm.temperature,
        )
        msg = resp.choices[0].message
        finish = resp.choices[0].finish_reason

        if resp.usage:
            llm._in += resp.usage.prompt_tokens
            llm._out += resp.usage.completion_tokens

        if finish == "stop":  # <2>
            return msg.content

        if finish == "tool_calls":
            messages.append(msg)  # append assistant message with tool_calls
            if verbose:
                for tc in msg.tool_calls:
                    print(f"  [step {step+1}] {tc.function.name}({tc.function.arguments})")

            for tool_call in msg.tool_calls:  # <3>
                result = registry.call(
                    tool_call.function.name,
                    tool_call.function.arguments,
                )
                if verbose:
                    print(f"           → {result[:120]}")
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })

    raise AgentTimeoutError(f"Agent exceeded {max_steps} steps without finishing.")

1. We call `client.chat.completions.create` directly (not `llm.complete`) so we can pass the `tools` parameter. The `llm` object is still used for token accounting.
2. `finish_reason == "stop"` means the model is satisfied with the information it has gathered and is ready to produce the final answer. We return `msg.content` directly.
3. A single model response can request multiple tool calls in parallel — the API returns them as a list. We execute each and append all results before calling the model again.

Running the agent on a compliance check query with verbose trace:

In [ ]:
print("Q: Is our CET1 ratio compliant with regulatory requirements?")
print("-" * 60)
answer = run_agent(
    question="Is our CET1 ratio compliant with regulatory requirements?",
    registry=registry,
    llm=llm,
    verbose=True,
)
print("\nFinal answer:")
print(answer)

## In-Context Memory

The agent accumulates context across steps because all previous messages are included in every call to the API. This is **in-context memory**: the model effectively "remembers" everything in the current conversation because it literally re-reads it on each step. We demonstrate multi-step accumulation with a two-part question where the second answer depends on the first:

In [ ]:
# A single agent call that requires two tool invocations whose results combine
answer = run_agent(
    question=(
        "How did investment banking revenues perform this year, "
        "and what is our LCR headroom above the regulatory minimum?"
    ),
    registry=registry,
    llm=llm,
    verbose=True,
)
print("\nFinal answer:")
print(answer)

**Why in-context memory is fragile at scale.** The context window grows by one assistant message + one or more tool messages per step. For a 10-step agent with average tool result sizes of 200 tokens, the final call sees roughly 2,000 extra tokens compared to the first. At $k$ steps, cost grows roughly as $O(k^2)$ because each subsequent call pays for all previous messages. Additionally:

- If the agent crashes mid-run, the context is lost and it must restart from step 1.
- The context limit (128k for GPT-4o) caps the maximum conversation length.
- Long contexts degrade model attention on earlier messages.

LangGraph in [notebook 08](/courses/llm-eng/08-agents-langgraph.html) solves these problems with a typed, persistent state object that lives outside the context window.

## Financial Agent Demo

We run all three multi-step financial queries to validate the full agent loop:

In [ ]:
queries = [
    "Is our CET1 ratio compliant with regulatory requirements?",
    "What is the VaR for a $10M portfolio with 15% annual volatility at 95% confidence?",
    "How did investment banking revenues perform and what is our LCR headroom?",
]

for q in queries:
    print(f"Q: {q}")
    try:
        ans = run_agent(q, registry=registry, llm=llm, verbose=False)
        print(f"A: {ans}")
    except AgentTimeoutError as e:
        print(f"TIMEOUT: {e}")
    print()

print(f"Total API cost: ${llm.total_cost:.5f}")

## Exercises

1. **Add a `search_news` tool.** Implement `search_news(ticker: str) -> str` that returns a mock news headline (e.g. `"GS Q3 earnings beat consensus by 8%"`). Register it in the tool registry and run the agent on: "What is the latest news about GS and how do their revenues compare year-over-year?"

2. **Implement a step counter logger.** Modify `run_agent` to accept an optional `log: list` parameter. On each tool invocation, append a dict `{"step": N, "tool": name, "args": kwargs, "result": result}` to the log. After the agent finishes, print the log as a numbered trace.

3. **Detect repeated tool calls.** Modify `run_agent` to track the last tool call per tool name. If the model calls the same tool with the identical arguments twice consecutively, raise a `AgentLoopError` with a descriptive message. This prevents infinite loops in cases where the model re-calls a tool after receiving an error result it cannot handle.

---

$\blacksquare$